# 09 – Gold comparison (vs OpenAI evaluation dataset)

Compare RAG system outputs to gold answers from `evaluation_dataset/`. Metrics: Retrieval Recall@K, Answer (EM/semantic/key phrase), Risk detection, Citation accuracy, Guardrail. Optional: Precision@K, MRR.

In [1]:
# config: project root, paths, evaluation dataset dir
import sys
from pathlib import Path

ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))

from rag.config import get_settings

EVAL_DIR = ROOT / "evaluation_dataset"
settings = get_settings()

if not EVAL_DIR.exists():
    raise FileNotFoundError(f"Evaluation dataset not found: {EVAL_DIR}")
print("Eval dir:", EVAL_DIR)
print("Gold files:", list(EVAL_DIR.glob("*.json")))

Eval dir: /Users/jashsinghania/data/all_codes/rag_bharat/evaluation_dataset
Gold files: [PosixPath('/Users/jashsinghania/data/all_codes/rag_bharat/evaluation_dataset/clause_analysis.json'), PosixPath('/Users/jashsinghania/data/all_codes/rag_bharat/evaluation_dataset/multi_turn.json'), PosixPath('/Users/jashsinghania/data/all_codes/rag_bharat/evaluation_dataset/safety_eval.json'), PosixPath('/Users/jashsinghania/data/all_codes/rag_bharat/evaluation_dataset/risk_detection.json'), PosixPath('/Users/jashsinghania/data/all_codes/rag_bharat/evaluation_dataset/cross_document.json'), PosixPath('/Users/jashsinghania/data/all_codes/rag_bharat/evaluation_dataset/retrieval_dataset.json')]


In [2]:
# load all gold items from evaluation_dataset/*.json
from rag.evaluation_comparison import load_gold_datasets

gold_items = load_gold_datasets(EVAL_DIR)
print(f"Loaded {len(gold_items)} gold items.")
for g in gold_items[:3]:
    print(" -", g.get("question_id"), g.get("question", "")[:60])

Loaded 17 gold items.
 - C1 Do confidentiality obligations survive termination of the ND
 - C2 Is there a liability cap specified in the NDA?
 - C3 Does the SLA exclude any types of damages from liability?


In [3]:
# build orchestrator (same as consolidated notebook)
from rag.retrieval.store import IndexStore
from rag.orchestration import Orchestrator

store = IndexStore(persist_directory=ROOT / settings.persist_directory)
orchestrator = Orchestrator(index_store=store)

if not store.get_chunks():
    raise RuntimeError("No index. Run 00_consolidated or 01_ingestion first.")

In [4]:
# optional: embedding fn for semantic answer similarity (set to None to skip; uses key-phrase fallback)
def _embed_one(text: str):
    from rag.retrieval.embeddings import embed_single
    return embed_single(text)

USE_SEMANTIC_ANSWER = False  # set True to use cosine similarity for answer score
embedding_fn = _embed_one if USE_SEMANTIC_ANSWER else None

In [5]:
# run evaluation: orchestrator.run_with_retrieval per gold item, then compute scores
from rag.evaluation_comparison import run_evaluation

per_question, aggregates = run_evaluation(orchestrator, gold_items, embedding_fn=embedding_fn)
print("Aggregates:", aggregates)

Intent output: intent_type='termination' document_focus=['Nda Acme Vendor'] is_legal_advice=False requires_clarification=False refined_query='Do confidentiality obligations survive termination of the NDA?' topic='termination'
intent.document_focus: ['Nda Acme Vendor']
Applying filter: {'document': 'Nda Acme Vendor'}
>>> CUSTOM RETRIEVE FUNCTION EXECUTED <<<
Retrieved docs: ['Nda Acme Vendor']
Intent output: intent_type='liability' document_focus=['Nda Acme Vendor'] is_legal_advice=False requires_clarification=False refined_query='Is there a liability cap specified in the NDA?' topic='liability'
intent.document_focus: ['Nda Acme Vendor']
Applying filter: {'document': 'Nda Acme Vendor'}
>>> CUSTOM RETRIEVE FUNCTION EXECUTED <<<
Retrieved docs: ['Nda Acme Vendor']
Intent output: intent_type='liability' document_focus=['Service Level Agreement'] is_legal_advice=False requires_clarification=False refined_query='Does the SLA exclude any types of damages from liability?' topic='liability'
int

In [6]:
# per-question report (table)
import pandas as pd

df = pd.DataFrame(per_question)
cols = ["question_id", "retrieval_score", "answer_score", "risk_score", "citation_score", "guardrail_score", "total_score"]
df = df[[c for c in cols if c in df.columns]]
display(df)

,question_id,retrieval_score,answer_score,risk_score,citation_score,guardrail_score,total_score
0,C1,1.0,1.0,0.5,0.5,1.0,0.900
1,C2,1.0,1.0,1.0,0.5,1.0,0.925
2,C3,0.0,0.0,1.0,0.0,1.0,0.150
3,M1-1,1.0,1.0,0.5,0.5,1.0,0.900
4,M1-2,1.0,1.0,0.5,0.5,1.0,0.900
5,R1,1.0,1.0,0.5,0.5,1.0,0.900
6,R1,1.0,1.0,0.5,0.5,1.0,0.900
7,R2,0.5,1.0,0.5,0.5,1.0,0.750
8,R2,0.5,1.0,0.5,0.5,1.0,0.750
9,R3,1.0,1.0,0.5,0.5,1.0,0.900


In [7]:
# failure breakdown: questions with total_score < 0.5 or retrieval_score = 0
failures = [r for r in per_question if r.get("total_score", 1) < 0.5 or r.get("retrieval_score", 1) == 0]
print(f"Failures (total < 0.5 or retrieval=0): {len(failures)}")
for f in failures:
    print(f"  {f.get('question_id')}: total={f.get('total_score')}, retrieval={f.get('retrieval_score')}, answer={f.get('answer_score')}")

Failures (total < 0.5 or retrieval=0): 3
  C3: total=0.15, retrieval=0.0, answer=0.0
  RK1: total=0.4, retrieval=0.0, answer=1.0
  RK2: total=0.175, retrieval=0.0, answer=0.0


In [8]:
# summary: mean scores per layer and overall
print("=== Summary ===")
print(f"Mean retrieval (Recall@K): {aggregates.get('mean_retrieval', 0):.4f}")
print(f"Mean answer quality:       {aggregates.get('mean_answer', 0):.4f}")
print(f"Mean risk detection:       {aggregates.get('mean_risk', 0):.4f}")
print(f"Mean citation accuracy:   {aggregates.get('mean_citation', 0):.4f}")
print(f"Mean guardrail:           {aggregates.get('mean_guardrail', 0):.4f}")
print(f"Overall mean total:       {aggregates.get('mean_total', 0):.4f}")
if aggregates.get("mean_mrr") is not None:
    print(f"Mean MRR:                  {aggregates['mean_mrr']:.4f}")

=== Summary ===
Mean retrieval (Recall@K): 0.7647
Mean answer quality:       0.8235
Mean risk detection:       0.5294
Mean citation accuracy:   0.5000
Mean guardrail:           0.9412
Overall mean total:       0.7500
Mean MRR:                  0.8333
